In [3]:
import zipfile
import pandas as pd

In [4]:
# Location of our Kaggle dataset
zip_path = "../data/SWPK.zip"

# Open the ZIP and list the CSV files inside
with zipfile.ZipFile(zip_path, "r") as zip_file:
    csv_files = [
        filename
        for filename in zip_file.namelist()
        if filename.endswith(".csv")
    ]

print("CSV files found:", len(csv_files))
print(csv_files)

CSV files found: 22
['datadump_s5-000.csv', 'datadump_s5-001.csv', 'datadump_s5-002.csv', 'datadump_s5-003.csv', 'datadump_s5-004.csv', 'datadump_s5-005.csv', 'datadump_s5-006.csv', 'datadump_s5-007.csv', 'datadump_s5-008.csv', 'datadump_s5-009.csv', 'datadump_s5-010.csv', 'datadump_s5-011.csv', 'datadump_s5-012.csv', 'datadump_s5-013.csv', 'datadump_s5-014.csv', 'datadump_s5-015.csv', 'datadump_s5-016.csv', 'datadump_s5-017.csv', 'datadump_s5-018.csv', 'datadump_s5-019.csv', 'datadump_s5-020.csv', 'datadump_s5-021.csv']


In [5]:
# Columns needed for our machine-learning project
needed_columns = [
    "matchid",
    "roundnumber",
    "mapname",
    "objectivelocation",
    "winrole",
    "role",
    "operator"
]

print("Columns we will use:")
print(needed_columns)

Columns we will use:
['matchid', 'roundnumber', 'mapname', 'objectivelocation', 'winrole', 'role', 'operator']


In [6]:
# Store the processed rounds from each CSV file
all_rounds = []

# Process each CSV file one at a time
with zipfile.ZipFile(zip_path, "r") as zip_file:

    for i, filename in enumerate(csv_files, start=1):

        print(f"Processing {i}/{len(csv_files)}: {filename}")

        # Open one CSV directly from the ZIP
        with zip_file.open(filename) as csv_file:

            df_part = pd.read_csv(
                csv_file,
                usecols=needed_columns
            )

        # Store the rounds found in this file
        file_rounds = []

        # Group players by match and round WITHIN this file
        for (matchid, roundnumber), group in df_part.groupby(
            ["matchid", "roundnumber"]
        ):

            # Get the five attackers
            attackers = group[
                group["role"] == "Attacker"
            ]["operator"].tolist()

            # Get the five defenders
            defenders = group[
                group["role"] == "Defender"
            ]["operator"].tolist()

            # Only keep complete 5v5 rounds
            if len(attackers) != 5 or len(defenders) != 5:
                continue

            # These values are the same for every player in the round
            mapname = group["mapname"].iloc[0]
            objectivelocation = group["objectivelocation"].iloc[0]
            winrole = group["winrole"].iloc[0]

            # 1 = attackers won
            # 0 = defenders won
            attack_win = 1 if winrole == "Attacker" else 0

            file_rounds.append({
                "source_file": filename,
                "matchid": matchid,
                "roundnumber": roundnumber,
                "mapname": mapname,
                "objectivelocation": objectivelocation,
                "attackers": attackers,
                "defenders": defenders,
                "attack_win": attack_win
            })

        # Convert this file's results into a DataFrame
        file_rounds = pd.DataFrame(file_rounds)

        print("Complete rounds:", len(file_rounds))

        all_rounds.append(file_rounds)


# Combine all 21 CSV files
round_df_full = pd.concat(
    all_rounds,
    ignore_index=True
)


# Clean operator names
def clean_operator_name(operator):
    operator_name = operator.split("-")[-1]

    # Historical dataset name for Recruit
    if operator_name == "RESERVE":
        return "RECRUIT"

    return operator_name


round_df_full["attackers"] = round_df_full["attackers"].apply(
    lambda operators: [
        clean_operator_name(operator)
        for operator in operators
    ]
)

round_df_full["defenders"] = round_df_full["defenders"].apply(
    lambda operators: [
        clean_operator_name(operator)
        for operator in operators
    ]
)


print("\n==============================")
print("FULL DATASET COMPLETE")
print("==============================")
print("Total complete rounds:", len(round_df_full))

Processing 1/22: datadump_s5-000.csv
Complete rounds: 237993
Processing 2/22: datadump_s5-001.csv
Complete rounds: 244039
Processing 3/22: datadump_s5-002.csv
Complete rounds: 231777
Processing 4/22: datadump_s5-003.csv
Complete rounds: 234736
Processing 5/22: datadump_s5-004.csv
Complete rounds: 238728
Processing 6/22: datadump_s5-005.csv
Complete rounds: 247232
Processing 7/22: datadump_s5-006.csv
Complete rounds: 236541
Processing 8/22: datadump_s5-007.csv
Complete rounds: 232612
Processing 9/22: datadump_s5-008.csv
Complete rounds: 228447
Processing 10/22: datadump_s5-009.csv
Complete rounds: 239224
Processing 11/22: datadump_s5-010.csv
Complete rounds: 234906
Processing 12/22: datadump_s5-011.csv
Complete rounds: 236870
Processing 13/22: datadump_s5-012.csv
Complete rounds: 231904
Processing 14/22: datadump_s5-013.csv
Complete rounds: 239322
Processing 15/22: datadump_s5-014.csv
Complete rounds: 236074
Processing 16/22: datadump_s5-015.csv
Complete rounds: 232071
Processing 17/22:

In [7]:
# Check whether each source file + match + round is unique

duplicates = round_df_full.duplicated(
    subset=["source_file", "matchid", "roundnumber"]
).sum()

print("Duplicate rounds:", duplicates)

Duplicate rounds: 0


In [14]:
import pyarrow as pa
import pyarrow.parquet as pq

# Convert the DataFrame to an Arrow table
table = pa.Table.from_pandas(
    round_df_full,
    preserve_index=False
)

# Save the table
pq.write_table(
    table,
    "../data/round_df_full.parquet"
)

print("Checkpoint saved successfully!")

Checkpoint saved successfully!


In [15]:
import pyarrow.parquet as pq

table = pq.read_table(
    "../data/round_df_full.parquet"
)

print("Rows in checkpoint:", table.num_rows)

Rows in checkpoint: 5082146


In [16]:
print("Total rounds:", len(round_df_full))
print("Unique matches:", round_df_full["matchid"].nunique())
print("Unique maps:", round_df_full["mapname"].nunique())
print("Unique sites:", round_df_full["objectivelocation"].nunique())

print("\nAttack vs Defense wins:")
print(round_df_full["attack_win"].value_counts())

print("\nWin percentages:")
print(
    round_df_full["attack_win"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Total rounds: 5082146
Unique matches: 1181156
Unique maps: 16
Unique sites: 142

Attack vs Defense wins:
attack_win
1    2543618
0    2538528
Name: count, dtype: int64

Win percentages:
attack_win
1    50.05
0    49.95
Name: proportion, dtype: float64


In [17]:
# Get every unique operator used by attackers
attack_operators = sorted(
    set(
        operator
        for operators in round_df_full["attackers"]
        for operator in operators
    )
)

# Get every unique operator used by defenders
defense_operators = sorted(
    set(
        operator
        for operators in round_df_full["defenders"]
        for operator in operators
    )
)

print("Attack operators:", len(attack_operators))
print(attack_operators)

print("\nDefense operators:", len(defense_operators))
print(defense_operators)

Attack operators: 16
['ASH', 'BLACKBEARD', 'BLITZ', 'BUCK', 'CAPITAO', 'FUZE', 'GLAZ', 'HIBANA', 'IQ', 'JACKAL', 'MONTAGNE', 'RECRUIT', 'SLEDGE', 'THATCHER', 'THERMITE', 'TWITCH']

Defense operators: 16
['BANDIT', 'CASTLE', 'CAVEIRA', 'DOC', 'ECHO', 'FROST', 'JAGER', 'KAPKAN', 'MIRA', 'MUTE', 'PULSE', 'RECRUIT', 'ROOK', 'SMOKE', 'TACHANKA', 'VALKYRIE']


In [18]:
from collections import Counter

attack_counts = Counter(
    operator
    for operators in round_df_full["attackers"]
    for operator in operators
)

defense_counts = Counter(
    operator
    for operators in round_df_full["defenders"]
    for operator in operators
)

print("Most common attackers:")
print(attack_counts.most_common(15))

print("\nMost common defenders:")
print(defense_counts.most_common(15))

Most common attackers:
[('ASH', 3232341), ('TWITCH', 2561153), ('HIBANA', 2557492), ('THERMITE', 2446272), ('FUZE', 2040477), ('JACKAL', 1983908), ('SLEDGE', 1946514), ('GLAZ', 1738656), ('THATCHER', 1622384), ('BUCK', 1216447), ('BLACKBEARD', 935225), ('MONTAGNE', 892365), ('CAPITAO', 781121), ('IQ', 767688), ('BLITZ', 364939)]

Most common defenders:
[('JAGER', 3494748), ('BANDIT', 2682877), ('CAVEIRA', 2191108), ('VALKYRIE', 2153075), ('MUTE', 1878548), ('SMOKE', 1732514), ('FROST', 1707721), ('ROOK', 1702372), ('PULSE', 1689234), ('MIRA', 1669322), ('DOC', 1330461), ('KAPKAN', 1019121), ('CASTLE', 1001782), ('ECHO', 577806), ('RECRUIT', 394830)]


In [19]:
print("All attacker operator counts:")
for operator, count in sorted(attack_counts.items()):
    print(f"{operator:12} {count:,}")

print("\nAll defender operator counts:")
for operator, count in sorted(defense_counts.items()):
    print(f"{operator:12} {count:,}")

All attacker operator counts:
ASH          3,232,341
BLACKBEARD   935,225
BLITZ        364,939
BUCK         1,216,447
CAPITAO      781,121
FUZE         2,040,477
GLAZ         1,738,656
HIBANA       2,557,492
IQ           767,688
JACKAL       1,983,908
MONTAGNE     892,365
RECRUIT      323,748
SLEDGE       1,946,514
THATCHER     1,622,384
THERMITE     2,446,272
TWITCH       2,561,153

All defender operator counts:
BANDIT       2,682,877
CASTLE       1,001,782
CAVEIRA      2,191,108
DOC          1,330,461
ECHO         577,806
FROST        1,707,721
JAGER        3,494,748
KAPKAN       1,019,121
MIRA         1,669,322
MUTE         1,878,548
PULSE        1,689,234
RECRUIT      394,830
ROOK         1,702,372
SMOKE        1,732,514
TACHANKA     185,211
VALKYRIE     2,153,075


In [20]:
# See how many unique map + site combinations exist

map_site = (
    round_df_full["mapname"]
    + "__"
    + round_df_full["objectivelocation"]
)

print("Unique map + site combinations:", map_site.nunique())

Unique map + site combinations: 160


In [21]:
# Show some examples

print(
    map_site.drop_duplicates()
    .sort_values()
    .head(30)
)

4                                BANK__ARCHIVES
74                             BANK__CEO_OFFICE
319           BANK__EXECUTIVE_LOUNGE-CEO_OFFICE
0                                 BANK__LOCKERS
320                     BANK__LOCKERS-CCTV_ROOM
5                               BANK__OPEN_AREA
2318                           BANK__STAFF_ROOM
1014                 BANK__STAFF_ROOM-OPEN_AREA
1323                      BANK__TELLER'S_OFFICE
322              BANK__TELLERS'_OFFICE-ARCHIVES
1320                                BANK__VAULT
34                       BARTLETT_U.__CLASSROOM
295              BARTLETT_U.__CLASSROOM-LIBRARY
490                        BARTLETT_U.__KITCHEN
69              BARTLETT_U.__KITCHEN-PIANO_ROOM
38                         BARTLETT_U.__LIBRARY
39                          BARTLETT_U.__LOUNGE
3556                   BARTLETT_U.__MAIN_OFFICE
37                      BARTLETT_U.__MODEL_HALL
296           BARTLETT_U.__READING_ROOM-LIBRARY
1269     BARTLETT_U.__ROWING_MUSEUM-TROP

In [22]:
# Make a copy of our full round dataset
features_df = round_df_full.copy()


# --------------------------------
# 1. ATTACKER OPERATOR FEATURES
# --------------------------------

for operator in attack_operators:
    features_df[f"ATTACK_{operator}"] = features_df["attackers"].apply(
        lambda team: int(operator in team)
    )


# --------------------------------
# 2. DEFENDER OPERATOR FEATURES
# --------------------------------

for operator in defense_operators:
    features_df[f"DEFENSE_{operator}"] = features_df["defenders"].apply(
        lambda team: int(operator in team)
    )


# --------------------------------
# 3. MAP FEATURES
# --------------------------------

map_features = pd.get_dummies(
    features_df["mapname"],
    prefix="MAP",
    dtype="int8"
)


# --------------------------------
# 4. MAP + SITE FEATURES
# --------------------------------

features_df["map_site"] = (
    features_df["mapname"]
    + "__"
    + features_df["objectivelocation"]
)

map_site_features = pd.get_dummies(
    features_df["map_site"],
    prefix="MAPSITE",
    dtype="int8"
)


# Add the categorical features
features_df = pd.concat(
    [
        features_df,
        map_features,
        map_site_features
    ],
    axis=1
)


print("Feature dataset shape:", features_df.shape)

Feature dataset shape: (5082146, 217)


In [23]:
print("Rows:", len(features_df))
print("Columns:", len(features_df.columns))

print("\nMemory usage:")
print(
    round(
        features_df.memory_usage(deep=True).sum() / (1024 ** 3),
        2
    ),
    "GB"
)

Rows: 5082146
Columns: 217

Memory usage:
4.61 GB


In [25]:
import pyarrow as pa
import pyarrow.parquet as pq

# Convert the feature DataFrame to an Arrow table
table = pa.Table.from_pandas(
    features_df,
    preserve_index=False
)

# Save the feature dataset
pq.write_table(
    table,
    "../data/features_df.parquet"
)

print("Feature checkpoint saved!")

Feature checkpoint saved!


In [26]:
table = pq.read_table(
    "../data/features_df.parquet"
)

print("Rows:", table.num_rows)
print("Columns:", table.num_columns)

Rows: 5082146
Columns: 217


In [27]:
# Create a unique match identifier
# matchid can appear in multiple source files.

features_df["unique_match_id"] = (
    features_df["source_file"].astype(str)
    + "_"
    + features_df["matchid"].astype(str)
)

print("Unique matches:", features_df["unique_match_id"].nunique())

Unique matches: 1184270


In [28]:
print("Total rounds:", len(features_df))
print(
    "Average rounds per match:",
    len(features_df) / features_df["unique_match_id"].nunique()
)

Total rounds: 5082146
Average rounds per match: 4.2913744331951325


In [29]:
from sklearn.model_selection import train_test_split

# Get each unique match exactly once
unique_matches = features_df["unique_match_id"].unique()

# Split the matches, not individual rounds
train_matches, test_matches = train_test_split(
    unique_matches,
    test_size=0.20,
    random_state=42
)

print("Training matches:", len(train_matches))
print("Testing matches:", len(test_matches))

Training matches: 947416
Testing matches: 236854


In [30]:
train_mask = features_df["unique_match_id"].isin(train_matches)
test_mask = features_df["unique_match_id"].isin(test_matches)

print("Training rounds:", train_mask.sum())
print("Testing rounds:", test_mask.sum())
print("Total:", train_mask.sum() + test_mask.sum())

Training rounds: 4066663
Testing rounds: 1015483
Total: 5082146


In [31]:
# Columns that are metadata or the target, not model inputs
non_feature_columns = [
    "source_file",
    "matchid",
    "roundnumber",
    "mapname",
    "objectivelocation",
    "attackers",
    "defenders",
    "attack_win",
    "unique_match_id",
    "map_site"
]

# Get the actual ML feature columns
feature_columns = [
    column
    for column in features_df.columns
    if column not in non_feature_columns
]

print("Number of ML features:", len(feature_columns))
print(feature_columns)

Number of ML features: 208
['ATTACK_ASH', 'ATTACK_BLACKBEARD', 'ATTACK_BLITZ', 'ATTACK_BUCK', 'ATTACK_CAPITAO', 'ATTACK_FUZE', 'ATTACK_GLAZ', 'ATTACK_HIBANA', 'ATTACK_IQ', 'ATTACK_JACKAL', 'ATTACK_MONTAGNE', 'ATTACK_RECRUIT', 'ATTACK_SLEDGE', 'ATTACK_THATCHER', 'ATTACK_THERMITE', 'ATTACK_TWITCH', 'DEFENSE_BANDIT', 'DEFENSE_CASTLE', 'DEFENSE_CAVEIRA', 'DEFENSE_DOC', 'DEFENSE_ECHO', 'DEFENSE_FROST', 'DEFENSE_JAGER', 'DEFENSE_KAPKAN', 'DEFENSE_MIRA', 'DEFENSE_MUTE', 'DEFENSE_PULSE', 'DEFENSE_RECRUIT', 'DEFENSE_ROOK', 'DEFENSE_SMOKE', 'DEFENSE_TACHANKA', 'DEFENSE_VALKYRIE', 'MAP_BANK', 'MAP_BARTLETT_U.', 'MAP_BORDER', 'MAP_CHALET', 'MAP_CLUB_HOUSE', 'MAP_COASTLINE', 'MAP_CONSULATE', 'MAP_FAVELAS', 'MAP_HEREFORD_BASE', 'MAP_HOUSE', 'MAP_KAFE_DOSTOYEVSKY', 'MAP_KANAL', 'MAP_OREGON', 'MAP_PLANE', 'MAP_SKYSCRAPER', 'MAP_YACHT', 'MAPSITE_BANK__ARCHIVES', 'MAPSITE_BANK__CEO_OFFICE', 'MAPSITE_BANK__EXECUTIVE_LOUNGE-CEO_OFFICE', 'MAPSITE_BANK__LOCKERS', 'MAPSITE_BANK__LOCKERS-CCTV_ROOM', 'MAPSITE_

In [32]:
# Target variable
y = features_df["attack_win"].astype("int8")

print("Target shape:", y.shape)

Target shape: (5082146,)


In [33]:
# Build the training and testing feature matrices

X_train = features_df.loc[train_mask, feature_columns]
X_test = features_df.loc[test_mask, feature_columns]

y_train = y.loc[train_mask]
y_test = y.loc[test_mask]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (4066663, 208)
X_test: (1015483, 208)
y_train: (4066663,)
y_test: (1015483,)


In [34]:
from scipy.sparse import csr_matrix

# Convert training and testing data to sparse matrices
X_train_sparse = csr_matrix(X_train)
X_test_sparse = csr_matrix(X_test)

print("X_train sparse shape:", X_train_sparse.shape)
print("X_test sparse shape:", X_test_sparse.shape)

X_train sparse shape: (4066663, 208)
X_test sparse shape: (1015483, 208)


In [35]:
print(
    "Training non-zero values:",
    X_train_sparse.nnz
)

print(
    "Testing non-zero values:",
    X_test_sparse.nnz
)

print(
    "Training sparsity:",
    round(
        1 - (
            X_train_sparse.nnz /
            (X_train_sparse.shape[0] * X_train_sparse.shape[1])
        ),
        4
    ) * 100,
    "%"
)

Training non-zero values: 48737678
Testing non-zero values: 12169792
Training sparsity: 94.24 %


In [36]:
from sklearn.linear_model import LogisticRegression

# Create the Logistic Regression model
model = LogisticRegression(
    max_iter=1000,
    solver="liblinear"
)

# Train the model
model.fit(
    X_train_sparse,
    y_train
)

print("Model training complete!")

Model training complete!


In [38]:
# Predict probabilities for the test set
test_probabilities = model.predict_proba(X_test_sparse)

print(test_probabilities[:5])

[[0.41371968 0.58628032]
 [0.42636226 0.57363774]
 [0.50071973 0.49928027]
 [0.45555486 0.54444514]
 [0.50085269 0.49914731]]


In [39]:
# Get the probability that the attackers win
attack_probabilities = test_probabilities[:, 1]

print("First 10 attack win probabilities:")
print(attack_probabilities[:10])

First 10 attack win probabilities:
[0.58628032 0.57363774 0.49928027 0.54444514 0.49914731 0.50426528
 0.61434465 0.60336105 0.44863589 0.54889338]


In [40]:
# Convert probabilities to percentages
attack_percentages = attack_probabilities * 100

print("First 10 attack win percentages:")
print(attack_percentages[:10])

First 10 attack win percentages:
[58.62803239 57.36377377 49.9280271  54.44451412 49.91473138 50.4265277
 61.43446522 60.33610483 44.86358863 54.88933778]


In [41]:
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

# Convert probabilities into predicted classes
test_predictions = (attack_probabilities >= 0.5).astype(int)

# Accuracy
accuracy = accuracy_score(
    y_test,
    test_predictions
)

# Log loss
logloss = log_loss(
    y_test,
    attack_probabilities
)

# Brier score
brier = brier_score_loss(
    y_test,
    attack_probabilities
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Log Loss: {logloss:.4f}")
print(f"Brier Score: {brier:.4f}")

Accuracy: 0.5479
Log Loss: 0.6857
Brier Score: 0.2463


In [42]:
print("Test rounds:", len(y_test))
print("Actual attack wins:", int(y_test.sum()))
print("Actual defense wins:", int((y_test == 0).sum()))

Test rounds: 1015483
Actual attack wins: 507659
Actual defense wins: 507824


In [43]:
# Look at the strongest positive and negative model coefficients

feature_coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": model.coef_[0]
})

feature_coefficients["absolute_value"] = (
    feature_coefficients["coefficient"].abs()
)

print("Features associated most strongly with Attack wins:")

display(
    feature_coefficients
    .sort_values("coefficient", ascending=False)
    .head(15)
)

print("Features associated most strongly with Defense wins:")

display(
    feature_coefficients
    .sort_values("coefficient", ascending=True)
    .head(15)
)

Features associated most strongly with Attack wins:


,feature,coefficient,absolute_value
158,MAPSITE_KAFE_DOSTOYEVSKY__KITCHEN_PREP-BAKERY,0.569497,0.569497
96,MAPSITE_CLUB_HOUSE__CCTV_ROOM-CASH_ROOM,0.514172,0.514172
120,MAPSITE_CONSULATE__VISA_OFFICE,0.421674,0.421674
177,MAPSITE_OREGON__REAR_STAGE-WATCH_TOWER,0.417541,0.417541
197,MAPSITE_YACHT__AKLARK_SUB_ROOM,0.375435,0.375435
144,MAPSITE_HOUSE__KID'S_BEDROOM,0.356328,0.356328
77,MAPSITE_BORDER__TELLERS,0.350893,0.350893
131,MAPSITE_FAVELAS__3F_PACKAGING_ROOM-2F_METH_LAB,0.331638,0.331638
87,MAPSITE_CHALET__MASTER_BEDROOM-OFFICE,0.328556,0.328556
147,MAPSITE_HOUSE__LIVING_ROOM,0.316311,0.316311


Features associated most strongly with Defense wins:


,feature,coefficient,absolute_value
97,MAPSITE_CLUB_HOUSE__CHURCH,-0.561043,0.561043
198,MAPSITE_YACHT__CAFETERIA,-0.507323,0.507323
58,MAPSITE_BANK__VAULT,-0.456779,0.456779
199,MAPSITE_YACHT__CAFETERIA-STAFF_DORMITORY,-0.414102,0.414102
178,MAPSITE_OREGON__SUPPLY,-0.379651,0.379651
110,MAPSITE_COASTLINE__2F_THEATER,-0.350849,0.350849
83,MAPSITE_CHALET__KITCHEN,-0.289201,0.289201
91,MAPSITE_CLUB_HOUSE__ARSENAL_ROOM,-0.282967,0.282967
113,MAPSITE_CONSULATE__ARCHIVES,-0.279972,0.279972
153,MAPSITE_KAFE_DOSTOYEVSKY__BAR_BACKSTORE,-0.277587,0.277587


In [44]:
# Find only map/site feature columns
location_features = [
    column
    for column in feature_columns
    if column.startswith("MAP_") or column.startswith("MAPSITE_")
]

print("Location features:", len(location_features))

Location features: 176


In [45]:
from sklearn.linear_model import LogisticRegression

location_model = LogisticRegression(
    max_iter=1000,
    solver="liblinear"
)

location_model.fit(
    X_train[location_features],
    y_train
)

location_probabilities = location_model.predict_proba(
    X_test[location_features]
)[:, 1]

location_predictions = (
    location_probabilities >= 0.5
).astype(int)

print(
    "Location-only accuracy:",
    accuracy_score(y_test, location_predictions)
)

print(
    "Location-only log loss:",
    log_loss(y_test, location_probabilities)
)

print(
    "Location-only Brier score:",
    brier_score_loss(y_test, location_probabilities)
)

Location-only accuracy: 0.5358474735667658
Location-only log loss: 0.68911726583237
Location-only Brier score: 0.24799459054935935


In [46]:
# Find only the operator feature columns
operator_features = [
    column
    for column in feature_columns
    if column.startswith("ATTACK_")
    or column.startswith("DEFENSE_")
]

print("Operator features:", len(operator_features))
print(operator_features)

Operator features: 32
['ATTACK_ASH', 'ATTACK_BLACKBEARD', 'ATTACK_BLITZ', 'ATTACK_BUCK', 'ATTACK_CAPITAO', 'ATTACK_FUZE', 'ATTACK_GLAZ', 'ATTACK_HIBANA', 'ATTACK_IQ', 'ATTACK_JACKAL', 'ATTACK_MONTAGNE', 'ATTACK_RECRUIT', 'ATTACK_SLEDGE', 'ATTACK_THATCHER', 'ATTACK_THERMITE', 'ATTACK_TWITCH', 'DEFENSE_BANDIT', 'DEFENSE_CASTLE', 'DEFENSE_CAVEIRA', 'DEFENSE_DOC', 'DEFENSE_ECHO', 'DEFENSE_FROST', 'DEFENSE_JAGER', 'DEFENSE_KAPKAN', 'DEFENSE_MIRA', 'DEFENSE_MUTE', 'DEFENSE_PULSE', 'DEFENSE_RECRUIT', 'DEFENSE_ROOK', 'DEFENSE_SMOKE', 'DEFENSE_TACHANKA', 'DEFENSE_VALKYRIE']


In [47]:
operator_model = LogisticRegression(
    max_iter=1000,
    solver="liblinear"
)

operator_model.fit(
    X_train[operator_features],
    y_train
)

operator_probabilities = operator_model.predict_proba(
    X_test[operator_features]
)[:, 1]

operator_predictions = (
    operator_probabilities >= 0.5
).astype(int)

print(
    "Operator-only accuracy:",
    accuracy_score(y_test, operator_predictions)
)

print(
    "Operator-only log loss:",
    log_loss(y_test, operator_probabilities)
)

print(
    "Operator-only Brier score:",
    brier_score_loss(y_test, operator_probabilities)
)

Operator-only accuracy: 0.5324746943080287
Operator-only log loss: 0.689818760180215
Operator-only Brier score: 0.24834190856027738


In [48]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=12,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42
)

rf_model.fit(
    X_train_sparse,
    y_train
)

print("Random Forest training complete!")

Random Forest training complete!


In [49]:
# Predict the probability that Attack wins
rf_probabilities = rf_model.predict_proba(
    X_test_sparse
)[:, 1]

# Convert probabilities into 0/1 predictions
rf_predictions = (
    rf_probabilities >= 0.5
).astype(int)

# Evaluate the model
rf_accuracy = accuracy_score(
    y_test,
    rf_predictions
)

rf_logloss = log_loss(
    y_test,
    rf_probabilities
)

rf_brier = brier_score_loss(
    y_test,
    rf_probabilities
)

print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest Log Loss: {rf_logloss:.4f}")
print(f"Random Forest Brier Score: {rf_brier:.4f}")

Random Forest Accuracy: 0.5408
Random Forest Log Loss: 0.6890
Random Forest Brier Score: 0.2479


In [50]:
print("MODEL COMPARISON")
print("----------------")
print(f"Logistic Regression Accuracy: {accuracy:.4f}")
print(f"Random Forest Accuracy:        {rf_accuracy:.4f}")
print()

print(f"Logistic Regression Log Loss:  {logloss:.4f}")
print(f"Random Forest Log Loss:        {rf_logloss:.4f}")
print()

print(f"Logistic Regression Brier:     {brier:.4f}")
print(f"Random Forest Brier:           {rf_brier:.4f}")

MODEL COMPARISON
----------------
Logistic Regression Accuracy: 0.5479
Random Forest Accuracy:        0.5408

Logistic Regression Log Loss:  0.6857
Random Forest Log Loss:        0.6890

Logistic Regression Brier:     0.2463
Random Forest Brier:           0.2479


In [51]:
print("features_df:", features_df.shape)
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

features_df: (5082146, 218)
X_train: (4066663, 208)
X_test: (1015483, 208)


In [52]:
# Create the 256 possible attacker-vs-defender matchup features

matchup_feature_names = []

for attacker in attack_operators:
    for defender in defense_operators:
        matchup_feature_names.append(
            f"MATCHUP_{attacker}_VS_{defender}"
        )

print("Number of matchup features:", len(matchup_feature_names))
print(matchup_feature_names[:10])
print("...")
print(matchup_feature_names[-10:])

Number of matchup features: 256
['MATCHUP_ASH_VS_BANDIT', 'MATCHUP_ASH_VS_CASTLE', 'MATCHUP_ASH_VS_CAVEIRA', 'MATCHUP_ASH_VS_DOC', 'MATCHUP_ASH_VS_ECHO', 'MATCHUP_ASH_VS_FROST', 'MATCHUP_ASH_VS_JAGER', 'MATCHUP_ASH_VS_KAPKAN', 'MATCHUP_ASH_VS_MIRA', 'MATCHUP_ASH_VS_MUTE']
...
['MATCHUP_TWITCH_VS_JAGER', 'MATCHUP_TWITCH_VS_KAPKAN', 'MATCHUP_TWITCH_VS_MIRA', 'MATCHUP_TWITCH_VS_MUTE', 'MATCHUP_TWITCH_VS_PULSE', 'MATCHUP_TWITCH_VS_RECRUIT', 'MATCHUP_TWITCH_VS_ROOK', 'MATCHUP_TWITCH_VS_SMOKE', 'MATCHUP_TWITCH_VS_TACHANKA', 'MATCHUP_TWITCH_VS_VALKYRIE']


In [53]:
# Create a sparse matrix for the 256 matchup features

from scipy.sparse import lil_matrix

matchup_matrix = lil_matrix(
    (len(features_df), len(matchup_feature_names)),
    dtype="int8"
)

for i, (attackers, defenders) in enumerate(
    zip(
        features_df["attackers"],
        features_df["defenders"]
    )
):
    for attacker in attackers:
        for defender in defenders:

            feature_name = f"MATCHUP_{attacker}_VS_{defender}"
            column_index = matchup_feature_names.index(feature_name)

            matchup_matrix[i, column_index] = 1

print("Matchup matrix shape:", matchup_matrix.shape)
print("Non-zero matchup values:", matchup_matrix.nnz)

Matchup matrix shape: (5082146, 256)
Non-zero matchup values: 126664516


In [54]:
from scipy.sparse import csr_matrix

matchup_matrix = csr_matrix(matchup_matrix)

print("Matchup matrix shape:", matchup_matrix.shape)
print("Non-zero values:", matchup_matrix.nnz)

Matchup matrix shape: (5082146, 256)
Non-zero values: 126664516


In [56]:
# Convert the pandas masks to NumPy boolean arrays
train_mask_np = train_mask.to_numpy()
test_mask_np = test_mask.to_numpy()

# Use the NumPy masks to split the sparse matchup matrix
matchup_train = matchup_matrix[train_mask_np]
matchup_test = matchup_matrix[test_mask_np]

print("Matchup training shape:", matchup_train.shape)
print("Matchup testing shape:", matchup_test.shape)

Matchup training shape: (4066663, 256)
Matchup testing shape: (1015483, 256)


In [57]:
from scipy.sparse import hstack

X_train_with_matchups = hstack(
    [X_train_sparse, matchup_train],
    format="csr"
)

X_test_with_matchups = hstack(
    [X_test_sparse, matchup_test],
    format="csr"
)

print("New training shape:", X_train_with_matchups.shape)
print("New testing shape:", X_test_with_matchups.shape)

New training shape: (4066663, 464)
New testing shape: (1015483, 464)


In [58]:
# Logistic Regression with operator matchup features

matchup_model = LogisticRegression(
    max_iter=1000,
    solver="liblinear"
)

matchup_model.fit(
    X_train_with_matchups,
    y_train
)

print("Matchup-enhanced model training complete!")

Matchup-enhanced model training complete!


In [59]:
# Predict attack win probabilities
matchup_probabilities = matchup_model.predict_proba(
    X_test_with_matchups
)[:, 1]

# Convert probabilities to predictions
matchup_predictions = (
    matchup_probabilities >= 0.5
).astype(int)

# Evaluate
matchup_accuracy = accuracy_score(
    y_test,
    matchup_predictions
)

matchup_logloss = log_loss(
    y_test,
    matchup_probabilities
)

matchup_brier = brier_score_loss(
    y_test,
    matchup_probabilities
)

print(f"Matchup Model Accuracy: {matchup_accuracy:.4f}")
print(f"Matchup Model Log Loss: {matchup_logloss:.4f}")
print(f"Matchup Model Brier Score: {matchup_brier:.4f}")

Matchup Model Accuracy: 0.5482
Matchup Model Log Loss: 0.6856
Matchup Model Brier Score: 0.2463


In [62]:
print("MODEL COMPARISON")
print("----------------")

print("Original Logistic Regression:")
print(f"  Accuracy: {accuracy:.4f}")
print(f"  Log Loss: {logloss:.4f}")
print(f"  Brier:    {brier:.4f}")

print("\nMatchup-Enhanced Logistic Regression:")
print(f"  Accuracy: {matchup_accuracy:.4f}")
print(f"  Log Loss: {matchup_logloss:.4f}")
print(f"  Brier:    {matchup_brier:.4f}")

MODEL COMPARISON
----------------
Original Logistic Regression:
  Accuracy: 0.5479
  Log Loss: 0.6857
  Brier:    0.2463

Matchup-Enhanced Logistic Regression:
  Accuracy: 0.5482
  Log Loss: 0.6856
  Brier:    0.2463


In [1]:
# --- Recovery cell 1: reload checkpoint and rebuild the match-safe split ---
import pyarrow.parquet as pq
import numpy as np
from sklearn.model_selection import train_test_split

table = pq.read_table("../data/features_df.parquet")
features_df = table.to_pandas()

print(len(features_df))
print(features_df.shape)

features_df["unique_match_id"] = (
    features_df["source_file"].astype(str)
    + "_"
    + features_df["matchid"].astype(str)
)

match_ids = features_df["unique_match_id"].astype(str).to_numpy()
unique_matches = np.unique(match_ids)

train_matches, test_matches = train_test_split(
    unique_matches,
    test_size=0.20,
    random_state=42
)

train_mask_np = np.isin(match_ids, train_matches)
test_mask_np = np.isin(match_ids, test_matches)

print("Training rounds:", train_mask_np.sum())
print("Testing rounds:", test_mask_np.sum())

5082146
(5082146, 217)
Training rounds: 4065996
Testing rounds: 1016150


In [2]:
# --- Recovery cell 2: rebuild the 208-feature list and X/y ---
operator_cols = [c for c in features_df.columns if c.startswith("ATTACK_") or c.startswith("DEFENSE_")]
map_cols = [c for c in features_df.columns if c.startswith("MAP_") and "__" not in c]
mapsite_cols = [c for c in features_df.columns if "__" in c]

feature_cols = operator_cols + map_cols + mapsite_cols
print("Feature count:", len(feature_cols))  # expect 208

X = features_df[feature_cols].to_numpy()
y = features_df["attack_win"].to_numpy()

X_train, X_test = X[train_mask_np], X[test_mask_np]
y_train, y_test = y[train_mask_np], y[test_mask_np]

from scipy.sparse import csr_matrix
X_train_sparse = csr_matrix(X_train)
X_test_sparse = csr_matrix(X_test)

print("Training shape:", X_train_sparse.shape)
print("Testing shape:", X_test_sparse.shape)

Feature count: 208
Training shape: (4065996, 208)
Testing shape: (1016150, 208)


In [3]:
# --- Recovery cell 3: retrain baseline logistic regression ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

model = LogisticRegression(max_iter=1000, solver="liblinear")
model.fit(X_train_sparse, y_train)

probabilities = model.predict_proba(X_test_sparse)[:, 1]
predictions = (probabilities >= 0.5).astype(int)

accuracy = accuracy_score(y_test, predictions)
logloss = log_loss(y_test, probabilities)
brier = brier_score_loss(y_test, probabilities)

print(f"Accuracy: {accuracy:.4f}")
print(f"Log Loss: {logloss:.4f}")
print(f"Brier Score: {brier:.4f}")

Accuracy: 0.5477
Log Loss: 0.6858
Brier Score: 0.2464


In [4]:
# --- Recovery cell 4: rebuild matchup features (dict lookup instead of list.index) ---
attacker_ops = sorted(set(op.replace("ATTACK_", "") for op in operator_cols if op.startswith("ATTACK_")))
defender_ops = sorted(set(op.replace("DEFENSE_", "") for op in operator_cols if op.startswith("DEFENSE_")))

matchup_feature_names = [
    f"MATCHUP_{a}_VS_{d}"
    for a in attacker_ops
    for d in defender_ops
]
matchup_index = {name: i for i, name in enumerate(matchup_feature_names)}

print("Number of matchup features:", len(matchup_feature_names))

Number of matchup features: 256


In [5]:
# --- Recovery cell 5: build the matchup sparse matrix (dict lookup, much faster) ---
from scipy.sparse import lil_matrix, csr_matrix

matchup_matrix = lil_matrix(
    (len(features_df), len(matchup_feature_names)),
    dtype="int8"
)

for i, (attackers, defenders) in enumerate(
    zip(features_df["attackers"], features_df["defenders"])
):
    for attacker in attackers:
        for defender in defenders:
            column_index = matchup_index[f"MATCHUP_{attacker}_VS_{defender}"]
            matchup_matrix[i, column_index] = 1

matchup_matrix = csr_matrix(matchup_matrix)
print("Matchup matrix shape:", matchup_matrix.shape)
print("Non-zero matchup values:", matchup_matrix.nnz)

Matchup matrix shape: (5082146, 256)
Non-zero matchup values: 126664516


In [6]:
# --- Recovery cell 6: split matchup matrix, combine, retrain matchup-enhanced model ---
from scipy.sparse import hstack

matchup_train = matchup_matrix[train_mask_np]
matchup_test = matchup_matrix[test_mask_np]

X_train_with_matchups = hstack([X_train_sparse, matchup_train], format="csr")
X_test_with_matchups = hstack([X_test_sparse, matchup_test], format="csr")

matchup_model = LogisticRegression(max_iter=1000, solver="liblinear")
matchup_model.fit(X_train_with_matchups, y_train)

matchup_probabilities = matchup_model.predict_proba(X_test_with_matchups)[:, 1]
matchup_predictions = (matchup_probabilities >= 0.5).astype(int)

matchup_accuracy = accuracy_score(y_test, matchup_predictions)
matchup_logloss = log_loss(y_test, matchup_probabilities)
matchup_brier = brier_score_loss(y_test, matchup_probabilities)

print(f"Matchup Model Accuracy: {matchup_accuracy:.4f}")
print(f"Matchup Model Log Loss: {matchup_logloss:.4f}")
print(f"Matchup Model Brier Score: {matchup_brier:.4f}")

Matchup Model Accuracy: 0.5480
Matchup Model Log Loss: 0.6857
Matchup Model Brier Score: 0.2463


In [7]:
# --- Calibration ---
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(
    y_test,
    matchup_probabilities,
    n_bins=10,
    strategy="uniform"
)

print("Predicted probability | Actual attack win rate")
for predicted, actual in zip(prob_pred, prob_true):
    print(f"{predicted:.3f}                | {actual:.3f}")

Predicted probability | Actual attack win rate
0.285                | 0.306
0.377                | 0.376
0.460                | 0.460
0.540                | 0.539
0.627                | 0.628
0.717                | 0.727


In [8]:
# --- Save models now that models/ exists ---
import joblib

joblib.dump(model, "../models/baseline_logreg.joblib")
joblib.dump(matchup_model, "../models/matchup_logreg.joblib")
joblib.dump(feature_cols, "../models/feature_cols.joblib")
joblib.dump(matchup_feature_names, "../models/matchup_feature_names.joblib")

print("Models saved.")

Models saved.
